In [ ]:
import re
import sys
from pathlib import Path

import networkx as nx
import spacy


# LOAD SPACY

MODEL = "en_core_web_sm"

try:
    nlp = spacy.load(MODEL)

except OSError:
    print(f"spaCy model '{MODEL}' is not installed.")
    print()
    print("Run this in Terminal:")
    print()
    print(f"python3 -m spacy download {MODEL}")

    sys.exit(1)


# DEPENDENCY TYPES

SUBJECT_DEPS = {
    "nsubj",
    "nsubjpass",
    "csubj",
    "csubjpass"
}

OBJECT_DEPS = {
    "obj",
    "dobj",
    "attr",
    "oprd",
    "dative"
}

# BASIC CLEANING

def clean_text(text):
    return " ".join(text.strip().split())


# GET FULL NOUN PHRASE

def noun_phrase(doc, token):
    """
    Turns something like:

        States

    into:

        The United States

    when spaCy recognizes it as one noun phrase.
    """

    if token is None:
        return "UNKNOWN"

    for chunk in doc.noun_chunks:

        if chunk.start <= token.i < chunk.end:
            return clean_text(chunk.text)

    return clean_text(token.text)


# FIND SUBJECT

def find_subject(predicate, inherited_subject=None):

    # Look for a subject attached directly to the verb
    for child in predicate.children:

        if child.dep_ in SUBJECT_DEPS:
            return child

    # Coordinated verbs can inherit a subject

    if inherited_subject is not None:
        return inherited_subject

    # Try the parent verb

    if predicate.dep_ == "conj":

        for child in predicate.head.children:

            if child.dep_ in SUBJECT_DEPS:
                return child

    return None

# PASSIVE AGENT

def find_passive_agent(predicate):
    """
    Example:

        Mary was slapped by Nora.

    Finds:

        Nora
    """

    for child in predicate.children:

        if (
            child.dep_ == "agent"
            or
            (
                child.dep_ == "prep"
                and child.lower_ == "by"
            )
        ):

            for grandchild in child.children:

                if grandchild.dep_ == "pobj":
                    return grandchild

    return None


# FIND OBJECT

def find_object(predicate):

    # Direct object first

    for child in predicate.children:

        if child.dep_ in OBJECT_DEPS:
            return child

    # Passive agent:

    agent = find_passive_agent(predicate)

    if agent is not None:
        return agent

    # Prepositional object fallback
  
    for child in predicate.children:

        if child.dep_ in {
            "prep",
            "agent"
        }:

            for grandchild in child.children:

                if grandchild.dep_ == "pobj":
                    return grandchild

    return None


# DETECT PASSIVE

def is_passive(predicate):

    for child in predicate.children:

        if child.dep_ == "auxpass":
            return True

        if child.dep_ in {
            "nsubjpass",
            "csubjpass"
        }:
            return True

    return False

# TRUTH VALUE

def detect_truthvalue(predicate):
    """
    Examples:

        Iran invaded Iraq.
        -> true

        Iran did not invade Iraq.
        -> false
    """

    for token in predicate.subtree:

        if token.dep_ == "neg":
            return "false"

        if token.lower_ in {
            "not",
            "n't",
            "never"
        }:
            return "false"

    return "true"

# MODIFIER

def detect_modifier(predicate):
    """
    WorldView has values such as:

        factual
        goal
        placeholder

    For normal statements we currently use:

        factual

    We can make this smarter later once your professor
    defines exactly how each modifier should be detected.
    """

    return "factual"

# TENSE

def detect_tense(predicate):

    auxiliaries = []

    for child in predicate.children:

        if child.dep_ in {
            "aux",
            "auxpass"
        }:

            auxiliaries.append(child)

    aux_lemmas = {
        token.lemma_.lower()
        for token in auxiliaries
    }

    aux_words = {
        token.lower_
        for token in auxiliaries
    }

    morph_tense = set(
        predicate.morph.get("Tense")
    )

    verb_form = set(
        predicate.morph.get("VerbForm")
    )

    aspect = set(
        predicate.morph.get("Aspect")
    )

    # INFINITIVE

    if "Inf" in verb_form:
        return "infinitive"


    # GERUND

    if "Ger" in verb_form:
        return "gerund"


    
    # BASIC TENSE

    if (
        "will" in aux_words
        or
        "shall" in aux_words
    ):

        base = "future"

    elif "Past" in morph_tense:

        base = "past"

    elif "Pres" in morph_tense:

        base = "present"

    else:

        # Sometimes the auxiliary contains the tense
        #
        # Example:
        #
        # has attacked
        # was attacked

        aux_tenses = set()

        for aux in auxiliaries:

            aux_tenses.update(
                aux.morph.get("Tense")
            )

        if "Past" in aux_tenses:

            base = "past"

        elif "Pres" in aux_tenses:

            base = "present"

        else:

            base = "present"


    # PERFECT

    perfect = (
        "have" in aux_lemmas
    )

    # PROGRESSIVE

    progressive = (
        "Prog" in aspect
        or
        (
            "be" in aux_lemmas
            and
            predicate.tag_ == "VBG"
        )
    )

    # PASSIVE

    passive = is_passive(predicate)


    # BUILD WORLDVIEW TENSE

    parts = [base]

    if perfect:
        parts.append("perfect")

    if progressive:
        parts.append("progressive")

    if passive:
        parts.append("passive")

    return "-".join(parts)


# RELATIONSHIP NAME

def relation_label(predicate):

    # Lemmatize:
    #
    # invaded -> invade
    # attacked -> attack
    # running -> run

    relation = predicate.lemma_.lower()

    # Passive:
    #
    # Nora slapped Mary
    #
    # vs.
    #
    # Mary was slapped by Nora
    #
    # WorldView-style:
    #
    # slap-by

    if is_passive(predicate):
        relation = f"{relation}-by"

    return relation


# FIND ALL VERBS

def find_predicates(sent):
    """
    Finds the main verb and coordinated verbs.

    Example:

        John attacked Iraq and captured Baghdad.

    Detects:

        attacked
        captured
    """

    root = sent.root

    predicates = []


    # NORMAL VERB

    if root.pos_ in {
        "VERB",
        "AUX"
    }:

        predicates.append(root)


    # COPULAR SENTENCE

    else:

        cop = None

        for child in root.children:

            if child.dep_ == "cop":
                cop = child
                break

        if cop is not None:
            predicates.append(cop)


    # COORDINATED VERBS

    for token in sent:

        if (
            token.dep_ == "conj"
            and
            token.pos_ in {
                "VERB",
                "AUX"
            }
        ):

            predicates.append(token)


    # REMOVE DUPLICATES

    seen = set()

    unique = []

    for token in sorted(
        predicates,
        key=lambda t: t.i
    ):

        if token.i not in seen:

            seen.add(token.i)

            unique.append(token)

    return unique


# PARSE ONE RELATIONSHIP

def parse_predicate(
    doc,
    sent,
    predicate,
    inherited_subject=None
):

    # COPULAR SENTENCE

    if predicate.dep_ == "cop":

        complement = predicate.head

        subject_token = find_subject(
            complement,
            inherited_subject
        )

        object_text = clean_text(
            complement.text
        )

        relation = predicate.lemma_.lower()

        truthvalue = detect_truthvalue(
            complement
        )

        modifier = detect_modifier(
            complement
        )

        tense = detect_tense(
            predicate
        )


    # --------------------------------------------------------
    # NORMAL VERB
    # --------------------------------------------------------

    else:

        subject_token = find_subject(
            predicate,
            inherited_subject
        )

        object_token = find_object(
            predicate
        )

        object_text = noun_phrase(
            doc,
            object_token
        )

        relation = relation_label(
            predicate
        )

        truthvalue = detect_truthvalue(
            predicate
        )

        modifier = detect_modifier(
            predicate
        )

        tense = detect_tense(
            predicate
        )


    # --------------------------------------------------------
    # SUBJECT
    # --------------------------------------------------------

    subject_text = noun_phrase(
        doc,
        subject_token
    )


    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    result = {

        "subject":
            subject_text,

        "relation":
            relation,

        "truthvalue":
            truthvalue,

        "modifier":
            modifier,

        "tense":
            tense,

        "object":
            object_text,

        "sentence":
            clean_text(sent.text)
    }

    return result, subject_token


# ============================================================
# PARSE NORMAL ENGLISH
# ============================================================

def parse_text(text):
    """
    Main NLP parser.

    Takes:

        Iran invaded Iraq.

    Returns:

        Iran
        invade
        true
        factual
        past
        Iraq
    """

    doc = nlp(text)

    relationships = []


    # Handle multiple sentences

    for sent in doc.sents:

        predicates = find_predicates(sent)

        inherited_subject = None


        for predicate in predicates:

            relation, found_subject = parse_predicate(
                doc,
                sent,
                predicate,
                inherited_subject
            )


            # Save the first subject so coordinated
            # verbs can reuse it.

            if (
                inherited_subject is None
                and
                found_subject is not None
            ):

                inherited_subject = found_subject


            relationships.append(
                relation
            )


    return relationships


# ============================================================
# FORMAT WORDS FOR WORLDVIEW
# ============================================================

def worldview_word(value):
    """
    Automatically quotes phrases with spaces.

    Example:

        The United States

    becomes:

        "The United States"
    """

    value = str(value)


    if value == "":
        return '""'


    needs_quotes = bool(

        re.search(
            r'[\s()";#]',
            value
        )

    )


    if needs_quotes:

        escaped = (

            value
            .replace(
                "\\",
                "\\\\"
            )
            .replace(
                '"',
                '\\"'
            )

        )

        return f'"{escaped}"'


    return value


# ============================================================
# CONVERT TO WORLDVIEW TEXT
# ============================================================

def to_worldview(relation):

    fields = [

        relation["subject"],

        relation["relation"],

        relation["truthvalue"],

        relation["modifier"],

        relation["tense"],

        relation["object"]

    ]


    formatted = []

    for field in fields:

        formatted.append(
            worldview_word(field)
        )


    return (
        "("
        +
        " ".join(formatted)
        +
        ")"
    )


# ============================================================
# ADD ONE RELATIONSHIP TO NETWORKX
# ============================================================

def add_relationship_to_graph(
    G,
    relation,
    relationship_number
):

    subject = relation["subject"]

    obj = relation["object"]

    verb = relation["relation"]


    # Unique relationship node
    #
    # relationship_1
    # relationship_2
    # relationship_3

    relation_node = (
        f"relationship_"
        f"{relationship_number}"
    )


    # ========================================================
    # SUBJECT NODE
    # ========================================================

    G.add_node(

        subject,

        Node_Type="concept",

        Label=subject
    )


    # ========================================================
    # RELATIONSHIP NODE
    # ========================================================

    G.add_node(

        relation_node,

        Node_Type="relationship",

        Label=verb,

        Truthvalue=
            relation["truthvalue"],

        Modifier=
            relation["modifier"],

        Tense=
            relation["tense"],

        Sentence=
            relation["sentence"]

    )


    # ========================================================
    # OBJECT NODE
    # ========================================================

    G.add_node(

        obj,

        Node_Type="concept",

        Label=obj

    )


    # ========================================================
    # SUBJECT -> RELATIONSHIP
    # ========================================================

    G.add_edge(

        subject,

        relation_node,

        Edge_Type="subject"

    )


    # ========================================================
    # RELATIONSHIP -> OBJECT
    # ========================================================

    G.add_edge(

        relation_node,

        obj,

        Edge_Type="object"

    )


# ============================================================
# BUILD COMPLETE NETWORKX GRAPH
# ============================================================

def build_graph(relationships):

    G = nx.DiGraph()


    for number, relation in enumerate(
        relationships,
        start=1
    ):

        add_relationship_to_graph(

            G,

            relation,

            number

        )


    return G


# ============================================================
# PRINT RESULTS
# ============================================================

def print_relationships(
    relationships
):

    if not relationships:

        print(
            "No relationship was detected."
        )

        return


    for number, relation in enumerate(
        relationships,
        start=1
    ):

        print()

        print(
            f"Relationship {number}"
        )

        print(
            "-" * 45
        )

        print(
            "Original:",
            relation["sentence"]
        )

        print()

        print(
            "Subject:      ",
            relation["subject"]
        )

        print(
            "Relationship: ",
            relation["relation"]
        )

        print(
            "Truthvalue:   ",
            relation["truthvalue"]
        )

        print(
            "Modifier:     ",
            relation["modifier"]
        )

        print(
            "Tense:        ",
            relation["tense"]
        )

        print(
            "Object:       ",
            relation["object"]
        )

        print()

        print(
            "WorldView:    ",
            to_worldview(relation)
        )


# ============================================================
# PARSE AND SAVE GRAPHML
# ============================================================

def parse_and_save(
    text,
    output_file="output.graphml"
):

    relationships = parse_text(
        text
    )


    # Print what was detected

    print_relationships(
        relationships
    )


    # Build NetworkX graph

    if relationships:

        G = build_graph(
            relationships
        )


        # Save GraphML

        nx.write_graphml(
            G,
            output_file
        )


        print()

        print(
            "Graph saved:"
        )

        print(
            Path(
                output_file
            ).resolve()
        )


    return relationships


# ============================================================
# INTERACTIVE MODE
# ============================================================

def interactive_mode():

    print()
    print(
        "========================================"
    )

    print(
        "WorldView Automatic Sentence Parser"
    )

    print(
        "========================================"
    )

    print()

    print(
        "Type any English sentence."
    )

    print(
        "Type 'quit' when finished."
    )

    print()


    counter = 1


    while True:

        text = input(
            "Sentence: "
        ).strip()


        # Stop program

        if text.lower() in {

            "quit",

            "exit",

            "q"

        }:

            print(
                "Parser stopped."
            )

            break


        if not text:
            continue


        # Every sentence gets its own
        # GraphML file.

        output_file = (

            f"output_"
            f"{counter}"
            f".graphml"

        )


        parse_and_save(

            text,

            output_file

        )


        counter += 1

        print()


# ============================================================
# START PROGRAM
# ============================================================

if __name__ == "__main__":

    interactive_mode()


WorldView Automatic Sentence Parser

Type any English sentence.
Type 'quit' when finished.


Relationship 1
---------------------------------------------
Original: ( We view true factual present process_as_a_dialogue_with_all_the_country )

Subject:       We
Relationship:  view
Truthvalue:    true
Modifier:      factual
Tense:         present
Object:        true factual present process_as_a_dialogue_with_all_the_country

WorldView:     (We view true factual present "true factual present process_as_a_dialogue_with_all_the_country")

Graph saved:
/Users/nickpellegri/Documents/Github/w0rldview/output_1.graphml

